<div align="right"><sub>Notebook 最終更新: 2026-03-26 16:31</sub></div>
<h1><strong>06. Gradio によるAIエージェントの処理過程の可視化</strong></h1>

前回（05回）で作成した「Writer（執筆者）」と「Editor（編集長）」による自己改善ループ（Iterative Refinement）は強力ですが、途中経過がログの文字としてダラダラ出力されるだけでは直感的に分かりづらいという課題がありました。

この回は、エージェントが内部でどのような「壁打ち（やり取り）」をしているかを、**Gradio** を使ってブラウザ上でスマートに確認できる UI を構築します。

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig
from src.ui import create_agent_ui

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 768, temp: float = 0.5):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

# 前回と同じ Writer と Editor の設定を準備します
writer_prompt = """
あなたはプロのライターです。ユーザーからのテーマについて、まずは標準的な解説記事を書いてください。
もし、Editor（編集長）から修正指示が来た場合は、そのフィードバックを全面的に取り入れて書き直してください。
"""
editor_prompt = """
あなたは非常に厳しい編集長です。
提出された文章を読み、以下の2つの基準が【両方とも】完全に満たされているか評価してください：
1. 読者が日常生活でイメージしやすい具体的な活用シーンが明確に含まれているか
2. 全体的にワクワクするような、未来への希望を感じる感情豊かなトーンになっているか

もし、1つでも不足していると感じた場合（特に初稿など）は、不足している点を具体的に指摘して書き直しを要求してください。
【重要】絶対に自分で文章を書き直さず、Writerへの修正指示（コメント）だけを簡潔に出力してください。

もし、提出された文章がすでに上記の条件を完璧に満たしていると判断した場合は、「誤りなし」と出力して承認してください。
"""
writer = RoleConfig(name="Writer", system_prompt=writer_prompt)
editor = RoleConfig(name="Editor", system_prompt=editor_prompt)

agent = LLMExecutorCriticAgent(llm_chat, role_configs=[writer, editor])
print('エージェントの準備が完了しました。')

## **1. エージェントUIの起動**
以下のセルを実行して，UIを立ち上げます。前回コンソールに出力されていた「初稿 → 編集長のダメ出し → 第2稿」というステップが、サイドバーに綺麗に格納されていることを確認してください。

テスト用クエリ例: `「完全自動運転タクシー」が普及した社会について、300字程度で解説記事を書いてください。`

In [ ]:
def run_agent_for_ui(query):
    final_answer, full_log, steps = agent.run_pipeline(query, max_iterations=2)
    
    # ログを Markdown の引用形式に整形してアコーディオンに表示しやすくする
    formatted_log = ""
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
    return final_answer, formatted_log

# Public URL (share=True) を発行してブラウザで確認します
ui = create_agent_ui(run_agent_for_ui)
ui.launch(share=True, debug=True)

## **2. 演習：別パターンのエージェントを作成する**

「書く人」と「直す人」の役割を自由に変えることで、全く異なる品質向上を目指すことができます。
例えば、**「熱血営業マン（Writer）」** が書いた企画書を、**「超・論理的なリスク管理者（Critic）」** が添削するループを作ってみましょう。

In [ ]:
salesman_prompt = """
あなたは熱血営業マンです！圧倒的な熱量と勢いで、商品の魅力を語ってください。
リスク管理者から細かい指摘を受けた場合は、熱量はそのままに、リスクへの対策をしっかり追加して文章を書き直してください。
"""

risk_manager_prompt = """
あなたは冷徹で理詰めのリスク管理者です。
提出された宣伝文を読み、以下の2点が満たされているか評価してください：
1. 「デメリットやリスク」が少なくとも1つ正直に記載されているか
2. そのリスクに対する「具体的な解決策やサポート体制」が明記されているか

勢いだけでなく、以上の安全面への配慮が不足している場合は、厳密に指摘して書き直しを求めてください。
【重要】絶対に自分で文章を書き直さず、修正指示のみを出力してください。
条件を満たしていれば「誤りなし」と出力してください。
"""

custom_roles = [
    RoleConfig(name="Salesman", system_prompt=salesman_prompt),
    RoleConfig(name="RiskManager", system_prompt=risk_manager_prompt)
]
custom_agent = LLMExecutorCriticAgent(llm_chat, role_configs=custom_roles)

def run_custom_agent_ui(query):
    final_answer, full_log, steps = custom_agent.run_pipeline(query, max_iterations=2)
    formatted_log = ""
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
    return final_answer, formatted_log

ui_custom = create_agent_ui(run_custom_agent_ui)
ui_custom.launch(share=True, debug=True)

# テスト用クエリ例: 「AIが全自動で家事をしてくれる夢のホームロボットを売るための宣伝文を書いて！」

## **まとめ**
- UIを通じてAIエージェントの処理過程を可視化することで、「AIが裏側でどうやって賢く修正を繰り返しているか」をユーザーに分かりやすく伝えることができます。
- これは、単に便利だからだけでなく、「AIシステムをデバッグ・改善する（どの役割のプロンプトを直すべきか特定する）」上で非常に重要なプロセスです。

次回（最終回）では、このエージェントに RAG（外部知識参照）を統合し、「Fact Finder（事実を検索して書く人）」と「Fact Checker（資料と矛盾がないか検証する人）」による究極のシステムを完成させます。